# Overfitting in Neural Networks

Overfitting occurs when a model learns the training data too thoroughly, including its noise and outliers, leading to poor performance on new, unseen data. This issue is particularly prevalent in neural networks, which have a high capacity to fit complex patterns. By effectively preventing overfitting, you ensure that the model generalizes well to new data, resulting in improved performance on validation and test sets.


# Preventing Overfitting

Preventing overfitting is crucial for enhancing model performance. 

Several techniques can be employed to mitigate overfitting, including **Dropout** and **Early Stopping**.


## Dropout and Early Stopping Techniques

Dropout and Early Stopping are two important techniques for improving the performance of neural networks:

- **Dropout**: This regularization technique randomly sets a subset of neurons to zero during training. By doing so, it helps the model generalize better and prevents reliance on any single neuron.

- **Early Stopping**: This technique halts training when the model's performance on a validation set begins to degrade, thereby preventing overfitting.


In this notebook, I'll illustrate how to implement techniques for preventing overfitting, along with brief explanations of how they work, in both TensorFlow/Keras and PyTorch. For simplicity, I will use the classic Iris dataset for classification.

### Steps:

- **Data Preparation**: The Iris dataset will be loaded, standardized, and split into training and testing sets.
- **Model Architecture**: A Sequential model will be constructed with two hidden layers, each followed by Batch Normalization and Dropout layers, culminating in an output layer with softmax activation.
- **Compilation**: The model will be compiled using the Adam optimizer and sparse categorical cross-entropy loss.
- **Callbacks**: Early stopping will be implemented to prevent overfitting.
- **Training**: The model will be trained with a validation split and early stopping in place.
- **Evaluation**: The model will be evaluated on the test set, and predictions will be made and assessed for accuracy.

This implementation ensures that the model is robust, effectively avoids overfitting, and undergoes thorough evaluation.


## 1. Neural Network Model Creation and Training Approach Using TensorFlow


In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a Sequential model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 output neurons for the 3 classes in the Iris dataset
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=100, batch_size=8, validation_split=0.2, callbacks=[early_stopping])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy}')

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = tf.argmax(y_pred, axis=1)

# Evaluate the model using accuracy_score
accuracy = accuracy_score(y_test, y_pred_classes)
print(f'Accuracy: {accuracy}')


2025-05-26 14:25:52.703772: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.4592 - loss: 1.5855 - val_accuracy: 0.4583 - val_loss: 1.0405
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4091 - loss: 1.6818 - val_accuracy: 0.6667 - val_loss: 0.9405
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5046 - loss: 1.2743 - val_accuracy: 0.7500 - val_loss: 0.8625
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6878 - loss: 0.8186 - val_accuracy: 0.7500 - val_loss: 0.7989
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7470 - loss: 0.6682 - val_accuracy: 0.7500 - val_loss: 0.7513
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7760 - loss: 0.6014 - val_accuracy: 0.7500 - val_loss: 0.7100
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6769 - loss: 0.7682 - val_accuracy: 0.7500 - val_loss: 0.6745
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8077 - loss: 0.5812 - val_accuracy: 0.7500 -


## 2. PyTorch Implementation

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

# Standardize the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create DataLoader for training and testing sets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Define the neural network model
class IrisNN(nn.Module):
    def __init__(self):
        super(IrisNN, self).__init__()
        self.fc1 = nn.Linear(4, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(64, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.bn1(x)
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.bn2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Instantiate the model, define the loss function and optimizer
model = IrisNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Early Stopping implementation
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0, path='checkpoint.pt', restore_best_weights=True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.restore_best_weights = restore_best_weights

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

    def restore_best_weights(self, model):
        if self.restore_best_weights:
            model.load_state_dict(torch.load(self.path))

# Training loop with Early Stopping
early_stopping = EarlyStopping(patience=10, verbose=True, restore_best_weights=True)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(test_loader)  # Average validation loss
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_loader):.4f}, Validation Loss: {val_loss:.4f}')      


Epoch 1/100, Training Loss: 1.1477, Validation Loss: 0.8765
Epoch 2/100, Training Loss: 0.6616, Validation Loss: 0.4918
Epoch 3/100, Training Loss: 0.5527, Validation Loss: 0.3338
Epoch 4/100, Training Loss: 0.4193, Validation Loss: 0.2500
Epoch 5/100, Training Loss: 0.4453, Validation Loss: 0.2052
Epoch 6/100, Training Loss: 0.4110, Validation Loss: 0.1801
Epoch 7/100, Training Loss: 0.4383, Validation Loss: 0.1606
Epoch 8/100, Training Loss: 0.3294, Validation Loss: 0.1430
Epoch 9/100, Training Loss: 0.4488, Validation Loss: 0.1321
Epoch 10/100, Training Loss: 0.3833, Validation Loss: 0.1301
Epoch 11/100, Training Loss: 0.3280, Validation Loss: 0.1378
Epoch 12/100, Training Loss: 0.3659, Validation Loss: 0.1339
Epoch 13/100, Training Loss: 0.3082, Validation Loss: 0.1181
Epoch 14/100, Training Loss: 0.3701, Validation Loss: 0.1138
Epoch 15/100, Training Loss: 0.2857, Validation Loss: 0.1079
Epoch 16/100, Training Loss: 0.3374, Validation Loss: 0.0943
Epoch 17/100, Training Loss: 0.36

### Explanation

Callbacks are tools that customize the training process by executing specific actions at various stages.

#### EarlyStopping Callback

The `EarlyStopping` callback helps prevent overfitting by stopping training when a monitored metric, like validation loss, stops improving. It tracks the validation loss during each epoch and halts training if there’s no improvement for a set number of epochs (patience). If enabled, it can restore the model to the best weights observed during training.

**Benefits:**
- Prevents overfitting by stopping training at the right time.
- Saves computational resources by halting unnecessary training.
- Ensures the final model has the best weights.

#### Dropout

**Dropout** is a regularization technique that randomly sets a fraction of neurons to zero during training. This reduces reliance on specific neurons and encourages the model to learn more robust features.

### How They Work Together

Using Early Stopping and Dropout together effectively prevents overfitting:
- **Dropout** creates a more robust model by reducing dependency on individual neurons.
- **Early Stopping** halts training when overfitting begins, ensuring good performance on the validation set.

### Other Techniques:

#### Data Augmentation
- Increases dataset diversity by creating modified versions of training data, which is especially useful for image data.

#### Regularization Techniques
- Add penalties to the loss function to prevent overfitting, such as L1 (Lasso) and L2 (Ridge) regularization.

#### Batch Normalization
- Normalizes activations for each batch, stabilizing and speeding up training while acting as a regularizer.

#### Cross-Validation
- Ensures the model generalizes well by splitting data into multiple folds for training and validation.


## Conclusion

Preventing overfitting is essential for enhancing model performance. Techniques such as Dropout, Early Stopping, Data Augmentation, Regularization, Batch Normalization, and Cross-Validation significantly improve a model's ability to generalize to unseen data.
